In [ ]:
import pandas as pd
import os

In [ ]:
traffic_data_path = "verkehr.csv"
traffic_data = pd.read_csv(traffic_data_path, sep=';')

plz_data_path = "plz_pop_data.csv"
plz_data = pd.read_csv(plz_data_path, sep=';')

# apply number to plz_data transformation to traffic_data (by appending column plz to traffic_data based on "Zst" == Number
plz_locality = "/home/thore/Downloads/plz_geocoord.csv"
df_plz_loc = pd.read_csv(plz_locality, sep=",", header=0)


In [ ]:

plz_data.rename({"index": "plz", "Einwohner:Innen": "inhab_plz", "Bevölkerungsdichte": "density", "Einwohner im 100km-Umkreis":"inhab_100"}, inplace=True, axis=1)
plz_data.head()

In [ ]:
traffic_data.rename({"Zst": "station_id"}, inplace=True, axis=1)
traffic_data.set_index("station_id", inplace=True)
traffic_data.head()

In [ ]:
df_plz_loc.head()

In [ ]:
import re
from pyproj import Transformer
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Initialize geocoder
geolocator = Nominatim(user_agent="plz_extractor")
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)

# Read file
with open('dat2.txt', 'r', encoding='utf-8') as f:
    content = f.read()

split = content.split("addBKGPoempel(")


# frop all lines that do not start with results
split = [entry for entry in split if entry.startswith("results")]
split = [entry.split("results, \"")[1].split(",<div")[0] for entry in split]

data = []
for entry in split:
    split_entry = entry.split(",")
    long = float(split_entry[0])
    lat = float(split_entry[1])
    # continue all that have blue in the name
    if "blue" in split_entry[3]:
        continue

    # get the number in the brackets

    number = re.search(r'\((\d+)\)', split_entry[3])
    if number:
        number = int(number.group(1))
    else:
        number = None
    data.append({
        "long_": long,
        "lat_": lat,
        "station_id": number
    })
    
html_data = pd.DataFrame(data)
html_data.head()


In [ ]:
html_data.set_index("station_id", inplace=True)

In [ ]:
55.490450, 6.996040 # 9283: 47.62934906314557, 10.820777735846123
60.97620, 5.9727620 # 1108: 53.891849595827296, 10.670326976201341

In [ ]:
from pyproj import Transformer

# Transformer from EPSG:25832 (UTM zone 32N) to EPSG:4326 (WGS84 lat/lon)
transformer = Transformer.from_crs("EPSG:25832", "EPSG:4326", always_xy=True)

# transform all "long_" and "lat_" columns to lon and lat using the transformer
def transform_coordinates(df):
    # Create new columns for transformed coordinates
    _df = pd.DataFrame()
    _df['lon'], _df['lat'] = transformer.transform(df['long_'].values, df['lat_'].values)
    _df.index = df.index
    return _df
# Apply the transformation
html_data = transform_coordinates(html_data)
# Save the transformed data to a new CSV file

html_data.head()

In [ ]:

# find closest plz to html_data and add plz to html_data
def find_closest_plz(row, df):
    # Calculate the distance between the row and all rows in df
    distances = ((df['lng'] - row['lon'])**2 + (df['lat'] - row['lat'])**2)**0.5
    # Find the index of the closest row
    closest_index = distances.idxmin()
    # Return the plz of the closest row
    return df.loc[closest_index, 'plz']
# Apply the function to each row in html_data
html_data['plz'] = html_data.apply(find_closest_plz, axis=1, df=df_plz_loc)
html_data.head()

In [ ]:
# append plz to traffic_data base on "Zst" == Number

html_data = html_data.reset_index(drop=True)
# 1. PLZ basierend auf "station_id" anhängen (station_id aus traffic_data, station_id aus html_data, both index)
traffic_data_plz = traffic_data.merge(
    html_data[['plz', 'lat', 'lon']],
    how='left',
    left_index=True,
    right_index=True
)

traffic_data_plz["station_id"] = traffic_data_plz.index

# 2. Einwohner basierend auf PLZ anhängen
traffic_data_plz_inhab = traffic_data_plz.merge(
    plz_data[['plz', 'inhab_plz', 'density', 'inhab_100']],
    how='left',
    on='plz'
)
traffic_data_plz_inhab.head()

In [ ]:

# Save the merged DataFrame to a new CSV file
output_path = "merged_traffic_data.csv"
traffic_data_plz_inhab.to_csv(output_path, sep=';', index=False)
print(f"Merged data saved to {output_path}")